In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import gym
from collections import deque

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(state_size, 24)
        self.fc2 = nn.Linear(24, 24)
        self.fc3 = nn.Linear(24, action_size)
    
        def forward(self, x):
            x = torch.relu(self.fc1(x))
            x = torch.relu(self.fc2(x))
            return self.fc3(x)
model = DQN(state_size, action_size)
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_function = nn.MSELoss()

memory = deque(maxlen=2000)
gamma = 0.95    #Discount factor
epsilon = 1.0   #Exploration rate
epsilon_min = 0.01
epsilon_decay = 0.995
batch_size = 64
episodes = 1000

def remember(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))

def reply():
    if len(memory) < batch_size:
        return

    minibatch = random.sample(memory, batch_size)
    for state, action, reward, next_state, done in minibatch:
        target = reward
        if not done:
            target = reward + gamma * torch.max(model(torch.FloatTensor(next_state))).item()

        output = model(torch.FloatTensor(state)) [action]

        #Ensure target has the same shape as output
        loss = loss_function(output.view(-1), torch.tensor([target], dtype=torch.float32))

        optimizer.zero_grad
        loss.backward()
        optimizer.step()

for episode in range(episodes):
    state = env.reset()[0]
    done = False
    total_reward = 0 

    while not done: 
        if np.random.rand() <= epsilon:
            action = random.randrange(action_size) #Explore
        else:
            action = torch.argmax(model(torch.FloatTensor(state))).item() #Exploit

        next_state, reward, done, _, _ = env.step(action)
        remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
    
    replay()
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    print(f"Episode {episode+1}, Total Reward: {total_reward}")
print("Training complete!")